<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 PCA: La Mejor Sombra de tus Datos 🔦
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/05%20-%20Reduccion%20de%20Dimensionalidad/Para%20Dummies/00_PCA_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 00 (PCA)](../00_PCA.ipynb). Cuando una tabla tiene **muchas columnas** (decenas, cientos), es difícil verla, dibujarla y hasta calcular con ella. *PCA* es un truco para **resumirla en unas pocas columnas nuevas** perdiendo lo menos posible. Lo vamos a entender con una historia cotidiana: **la sombra de un objeto**.

Al terminar podrás explicar:
1. **Qué busca PCA**: el «ángulo de luz» cuya sombra conserva más información.
2. **Cuántas columnas nuevas** conviene guardar (y por qué no existe una respuesta única).
3. Cómo **comprimir una imagen** con PCA y cuánto se pierde.
4. Las dos **trampas** más comunes: no igualar las escalas y esperar que PCA entienda formas curvas.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../00_PCA.ipynb). Otras miradas sencillas: [PCA (Dummies) en Data Mining](../../../Data%20Mining/01%20-%20Preprocesamiento%20de%20los%20Datos/Para%20Dummies/04_PCA_Dummies.ipynb) y [PCA (Dummies) en Data Science Programming](../../../Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/04_PCA_Feature_Engineering_Dummies.ipynb).

---
## 1. La sombra de un objeto 🔦

Imagina una escultura alargada, como un lápiz. Si la alumbras **de punta**, su sombra es un puntito: perdiste casi todo. Si la alumbras **de lado**, la sombra es larga y se ve bien cómo es el lápiz. *PCA* hace lo mismo con tus datos: **busca el ángulo de luz cuya sombra se estira más**, porque una sombra muy estirada es una sombra que **conserva más información**.

Un ejemplo con notas: 120 estudiantes con su nota de **matemáticas** y de **física** (de 0 a 5). Como quien va bien en una suele ir bien en la otra, los puntos forman una nube alargada y inclinada. El «ángulo de luz» ideal es **a lo largo de esa nube**: es el *primer componente* (algo así como «habilidad para las ciencias exactas»).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

rng = np.random.RandomState(42)
mate = rng.normal(3.5, 0.7, 120)                                 # nota de matematicas
fisica = 0.8 * mate + rng.normal(0.6, 0.25, 120)                 # la de fisica depende (en parte) de la de mate
notas = np.c_[mate, fisica]

pca = PCA(n_components=2).fit(notas)
centro = notas.mean(axis=0)
mejor = pca.components_[0]                                       # direccion de la mejor "sombra"
peor = pca.components_[1]                                        # la direccion perpendicular (la peor sombra)

def estiramiento(direccion):
    """Cuanto se estira la sombra de las notas sobre una direccion (desviacion estandar de la proyeccion)."""
    return ((notas - centro) @ direccion).std()

print(f"Sombra sobre el eje 'solo matemáticas': se estira {estiramiento(np.array([1.0, 0.0])):.2f}")
print(f"Sombra sobre el eje 'solo física'     : se estira {estiramiento(np.array([0.0, 1.0])):.2f}")
print(f"Sombra sobre la MEJOR dirección       : se estira {estiramiento(mejor):.2f}   <- la más larga posible")
print(f"Sombra sobre la PEOR dirección        : se estira {estiramiento(peor):.2f}")

plt.figure(figsize=(7, 5.5))
plt.scatter(notas[:, 0], notas[:, 1], s=18, alpha=0.6, color="#2563eb")
for d, color, nombre in [(mejor, "#dc2626", "mejor ángulo de luz (1.er componente)"), (peor, "#16a34a", "peor ángulo (2.º componente)")]:
    plt.plot(*np.c_[centro - 2.2 * d, centro + 2.2 * d], color=color, lw=3, label=nombre)
plt.xlabel("Nota de matemáticas"); plt.ylabel("Nota de física"); plt.axis("equal")
plt.title("PCA busca el eje a lo largo del cual la nube se estira más", fontweight="bold"); plt.legend(); plt.show()

---
## 2. ¿Cuántas «sombras» me quedo? ✂️

Con 2 columnas es fácil. Pero un dibujo de **64 píxeles** (las imágenes de dígitos de `scikit-learn`) tiene **64 columnas**. PCA ordena las columnas nuevas de la **más informativa a la menos informativa**: la primera captura lo más importante, la segunda lo siguiente, y así. Entonces la pregunta es práctica: **¿cuántas conservo?** Se mira cuánto de la información total («varianza») acumulan las primeras. Casi nunca hay un número exacto: depende de si quieres **dibujar** (2 o 3), **ahorrar espacio** o **alimentar a otro modelo**.

In [ ]:
from sklearn.datasets import load_digits

digitos = load_digits().data                                     # 1797 dibujos de 64 pixeles
pca64 = PCA().fit(digitos)
porcentaje = pca64.explained_variance_ratio_ * 100               # % de informacion de cada columna nueva
acumulado = np.cumsum(porcentaje)
n90 = int(np.searchsorted(acumulado, 90) + 1)

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(range(1, 16), porcentaje[:15], color="#2563eb", alpha=0.8, label="% que aporta cada columna nueva")
ax.set_xlabel("Columna nueva (componente)"); ax.set_ylabel("% de la información")
ax2 = ax.twinx(); ax2.plot(range(1, 16), acumulado[:15], "o-", color="#dc2626", label="% acumulado"); ax2.set_ylabel("% acumulado"); ax2.grid(False)
ax.set_title("Las primeras columnas nuevas concentran la información", fontweight="bold")
fig.legend(loc="center right", bbox_to_anchor=(0.88, 0.5)); plt.show()
print(f"De 64 columnas originales, con solo {n90} columnas nuevas guardamos el 90 % de la información.")
print(f"Con las 2 primeras (las que se pueden dibujar en un plano) guardamos {acumulado[1]:.0f} %.")

---
## 3. Comprimir una foto 🖼️

Si guardas solo unas pocas columnas nuevas por dibujo, **ahorras espacio**; para ver el dibujo otra vez, PCA lo **reconstruye** con lo que guardó. Cuantas más columnas guardes, **más nítida** sale la reconstrucción. Probemos con un mismo dígito.

In [ ]:
x = digitos[7:8]                                                 # un dibujo cualquiera (un 7)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
axes[0].imshow(x.reshape(8, 8), cmap="gray_r"); axes[0].set_title("original\n(64 números)", fontsize=9)
for ax, m in zip(axes[1:], [2, 5, 10, 20, 40]):
    p = PCA(n_components=m).fit(digitos)
    reconstruido = p.inverse_transform(p.transform(x))
    ax.imshow(reconstruido.reshape(8, 8), cmap="gray_r", vmin=0, vmax=16); ax.set_title(f"con {m} números", fontsize=9)
for ax in axes: ax.axis("off")
plt.suptitle("Con pocas columnas nuevas ya se adivina el dígito; con más, se parece cada vez más al original", fontweight="bold"); plt.tight_layout(); plt.show()

---
## 4. Dos trampas ⚠️

**Trampa 1: la regla y la báscula.** PCA busca dónde hay **más dispersión**. Si una columna se mide en números **enormes** (por ejemplo, el sueldo en pesos) y otra en números **pequeños** (la edad en años), PCA creerá que el sueldo es lo único importante, **solo porque sus números son grandes**. La solución: **igualar las escalas** (estandarizar) antes de aplicar PCA. Lo vemos con 13 medidas químicas de vinos: una de ellas (la *prolina*) tiene números gigantes.

**Trampa 2: solo entiende rectas.** PCA gira y proyecta con **líneas rectas**. Si tus datos forman un **rizo, un aro o un rollo**, ninguna sombra recta los «desenrolla». Para eso existen otros métodos, que veremos en los siguientes cuadernos ([t-SNE](01_tSNE_Dummies.ipynb) e [Isomap](02_Isomap_Variedades_Dummies.ipynb)).

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

vinos = load_wine()
sin_igualar = PCA(n_components=2).fit(vinos.data)
igualado = PCA(n_components=2).fit(StandardScaler().fit_transform(vinos.data))

dominante = vinos.feature_names[int(np.abs(sin_igualar.components_[0]).argmax())]
print(f"SIN igualar escalas: la 1.ª columna nueva guarda {sin_igualar.explained_variance_ratio_[0]*100:.1f} % ... y es casi puro '{dominante}'.")
print(f"CON escalas iguales: la 1.ª guarda {igualado.explained_variance_ratio_[0]*100:.1f} % y la 2.ª {igualado.explained_variance_ratio_[1]*100:.1f} %: la información se reparte de verdad.")

plt.figure(figsize=(7, 4.2))
plt.bar(["sin igualar escalas", "con escalas iguales"], [sin_igualar.explained_variance_ratio_[0] * 100, igualado.explained_variance_ratio_[0] * 100], color=["#dc2626", "#2563eb"])
plt.ylabel("% de información de la 1.ª columna nueva"); plt.title("Sin igualar escalas, una sola medida se roba el protagonismo", fontweight="bold"); plt.show()

---
##### 🛠️ Práctica 1: ¿Cuántas sombras para guardar el 95 %?

En la sección 2 contamos cuántas «sombras» (columnas nuevas) hacían falta para guardar el **90 %** de la información de los dibujos de dígitos (`n90`). Ahora vas a ser más exigente y ver cuánto cuesta:

1. Usa la lista `acumulado` (el % acumulado de información) y busca cuántas columnas nuevas hacen falta para llegar al **95 %**. Guarda ese número en `n95`.
2. Arma un PCA con esa cantidad de columnas (`PCA(n_components=n95)`), ajústalo con `digitos`, y reconstruye el dibujo `x` (el 7 de la sección 3). Guarda el resultado en `reconstruido`.
3. Calcula cuánto se equivoca la reconstrucción en promedio (la diferencia absoluta media entre `x` y `reconstruido`) y guárdalo en `error`.
4. Comprueba que `n95` es mayor que `n90` y que el PCA guardó de verdad al menos el 95 %. Imprime una frase con tus resultados.

In [ ]:
# Escribe tu código aquí

# 1) Cuantas columnas nuevas para llegar al 95 %
# n95 = int(np.searchsorted(acumulado, ...) + 1)

# 2) PCA con esas columnas y reconstruccion del dibujo x
# pca95 = PCA(n_components=...).fit(digitos)
# reconstruido = pca95.inverse_transform(pca95.transform(...))

# 3) Cuanto se equivoca en promedio
# error = np.abs(x - ...).mean()

# 4) Comprobaciones
# assert n95 > n90
# assert pca95.explained_variance_ratio_.sum() >= 0.95
# print(...)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Primera posicion donde el acumulado llega a 95 %, mas 1 (las posiciones empiezan en 0)
n95 = int(np.searchsorted(acumulado, 95) + 1)

# 2) PCA con esas columnas nuevas y reconstruccion del dibujo x
pca95 = PCA(n_components=n95).fit(digitos)
reconstruido = pca95.inverse_transform(pca95.transform(x))

# 3) Error promedio por pixel (los pixeles van de 0 a 16)
error = np.abs(x - reconstruido).mean()

# 4) Comprobaciones
assert n95 > n90
assert pca95.explained_variance_ratio_.sum() >= 0.95
print(f"Para el 90 % bastaban {n90} columnas; para el 95 % hacen falta {n95} de 64.")
print(f"Con {n95} columnas el dibujo se reconstruye con un error promedio de {error:.2f} (en una escala de 0 a 16).")
```
</details>

---


---
## Resumen en una frase 🎯

> ***PCA* resume una tabla con muchas columnas en unas pocas columnas nuevas, eligiendo los «ángulos de luz» cuya sombra se estira más (así conserva más información); conviene igualar las escalas antes y recordar que solo trabaja con rectas.**

### Lo que aprendiste hoy:
- ✅ PCA busca el **ángulo de luz** con la **sombra más larga** (más información).
- ✅ Ordena las columnas nuevas de **más a menos informativa**; **cuántas** guardar depende de tu objetivo.
- ✅ Sirve para **comprimir** (guardar menos números) y reconstruir con algo de pérdida.
- ✅ **Igualar las escalas** primero, o la medida con números más grandes domina todo.
- ✅ Solo entiende **rectas**: con formas curvas se necesitan otros métodos.

> 🎓 **Siguiente paso:** [t-SNE (Dummies)](01_tSNE_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../00_PCA.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué significa que una «sombra se estire más» y por qué eso se relaciona con conservar información?
2. Tienes una tabla de clientes con la columna *ingresos* (en pesos, millones) y la columna *edad* (en años). Si aplicas PCA sin hacer nada más, ¿qué columna dominaría y por qué? ¿Qué harías antes?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>